In [0]:

# ==========================================
# STEP 1: IMPORT REQUIRED PYSPARK FUNCTIONS
# ==========================================
from pyspark.sql.functions import (
    col, from_json, schema_of_json, explode, posexplode, 
    current_timestamp, to_date, when, size, lit, regexp_extract, concat_ws
)

# ==========================================
# STEP 2: DEFINE S3 PATHS & TABLE NAMES
# ==========================================
bronze_table_name = "workspace.default.bronze_ipl_matches"
silver_s3_path = "s3://garvit-ipl-data-lake/silver/deliveries"
silver_checkpoint_path = "s3://garvit-ipl-data-lake/logs/checkpoints/silver"
silver_table_name = "workspace.default.silver_ipl_ball_events"

# ==========================================
# STEP 3: AUTOMATIC SCHEMA DETECTION
# ==========================================
sample_json_string = spark.table(bronze_table_name).select("raw_json").first()[0]
match_schema = schema_of_json(sample_json_string)

# ==========================================
# STEP 4: READ STREAM & DECODE RAW JSON
# ==========================================
bronze_stream_df = spark.readStream.table(bronze_table_name)

parsed_matches_df = (
    bronze_stream_df
    .withColumn("match_data", from_json(col("raw_json"), match_schema))
    .withColumn("match_id", regexp_extract(col("source_file"), r"(\d+)\.json$", 1))
)

# ==========================================
# STEP 5: UNNEST INNINGS (1 Match -> 2 Rows)
# ==========================================
innings_level_df = parsed_matches_df.select(
    col("match_id"),
    col("match_data.info.season").alias("season"),
    to_date(col("match_data.info.dates")[0]).alias("match_date"),
    col("match_data.info.venue").alias("venue"),
    col("match_data.info.teams").alias("teams_array"),
    explode(col("match_data.innings")).alias("single_inning")
)

# ==========================================
# STEP 6: UNNEST OVERS (1 Inning -> 20 Overs)
# ==========================================
overs_level_df = innings_level_df.select(
    col("match_id"),
    col("season"),
    col("match_date"),
    col("venue"),
    col("teams_array"),
    col("single_inning.team").alias("batting_team"),
    explode(col("single_inning.overs")).alias("single_over")
)

# ==========================================
# STEP 7: UNNEST DELIVERIES (1 Over -> 6 Balls)
# ==========================================
deliveries_level_df = overs_level_df.select(
    col("match_id"),
    col("season"),
    col("match_date"),
    col("venue"),
    col("teams_array"),
    col("batting_team"),
    col("single_over.over").alias("over_number"),
    posexplode(col("single_over.deliveries")).alias("ball_position", "ball_data")
)

# ==========================================
# STEP 8: BUILD FINAL CLEAN COLUMNS
# ==========================================
final_silver_df = (
    deliveries_level_df
    .select(
        # 1. Unique Primary Key: MatchID_BattingTeam_Over_Ball Number
        concat_ws("_", col("match_id"), col("batting_team"), col("over_number"), col("ball_position") + 1).alias("delivery_id"),
        col("match_id"),
        col("season"),
        col("match_date"),
        col("venue"),
        col("batting_team"),
        
        # 2. Derive Bowling Team (Opposite of Batting Team)
        when(col("teams_array")[0] == col("batting_team"), col("teams_array")[1])
        .otherwise(col("teams_array")[0])
        .alias("bowling_team"),
        
        col("over_number"),
        (col("ball_position") + 1).alias("ball_number"),
        
        # 3. Player Names
        col("ball_data.batter").alias("batter"),
        col("ball_data.bowler").alias("bowler"),
        col("ball_data.non_striker").alias("non_striker"),
        
        # 4. Runs Breakup
        col("ball_data.runs.batter").alias("runs_batter"),
        col("ball_data.runs.extras").alias("runs_extras"),
        col("ball_data.runs.total").alias("runs_total"),
        
        # 5. Wicket Information
        when(size(col("ball_data.wickets")) > 0, lit(True)).otherwise(lit(False)).alias("is_wicket"),
        col("ball_data.wickets")[0]["kind"].alias("wicket_type"),
        col("ball_data.wickets")[0]["player_out"].alias("player_out"),
        
        current_timestamp().alias("silver_loaded_at")
    )
    # 6. Data Quality Filter: Remove Invalid/Null Records
    .filter(col("season").isNotNull())
    .filter(col("batting_team").isNotNull())
    .filter(col("batter").isNotNull())
    .filter(col("bowler").isNotNull())
    .filter(col("runs_total") >= 0)
)

# ==========================================
# STEP 9: WRITE STREAM TO S3 DELTA TABLE
# ==========================================
silver_stream_query = (
    final_silver_df.writeStream
                   .format("delta")
                   .outputMode("append")
                   .option("checkpointLocation", silver_checkpoint_path)
                   .option("path", silver_s3_path)
                   .option("tableName", silver_table_name)
                   .trigger(availableNow=True)
                   .start()
)

silver_stream_query.awaitTermination()
print("✅ Silver transformation completed successfully!")

✅ Silver transformation completed successfully!


In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.default.silver_ipl_ball_events
USING DELTA
LOCATION 's3://garvit-ipl-data-lake/silver/deliveries';

In [0]:
%sql
-- 1. Check Total Delivery Count
SELECT COUNT(*) AS total_deliveries FROM workspace.default.silver_ipl_ball_events;

total_deliveries
295732


In [0]:
%sql
SELECT delivery_id, season, batting_team, bowling_team, batter, bowler, runs_total 
FROM workspace.default.silver_ipl_ball_events 
LIMIT 5;

delivery_id,season,batting_team,bowling_team,batter,bowler,runs_total
1473460_Gujarat Titans_0_1,2025,Gujarat Titans,Rajasthan Royals,B Sai Sudharsan,JC Archer,0
1473460_Gujarat Titans_0_2,2025,Gujarat Titans,Rajasthan Royals,B Sai Sudharsan,JC Archer,0
1473460_Gujarat Titans_0_3,2025,Gujarat Titans,Rajasthan Royals,B Sai Sudharsan,JC Archer,1
1473460_Gujarat Titans_0_4,2025,Gujarat Titans,Rajasthan Royals,Shubman Gill,JC Archer,1
1473460_Gujarat Titans_0_5,2025,Gujarat Titans,Rajasthan Royals,Shubman Gill,JC Archer,1
